# Stage 3 — Unsupervised Clustering and Model Evaluation

## Objective

Investigate whether meaningful research themes can be identified in arXiv NLP publications (filtered by cs.CL category and 2021–2025 years) using unsupervised clustering of titles and abstracts.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import joblib

artifact_dir = Path("artifacts")

X_tfidf = sparse.load_npz(
    artifact_dir / "tfidf_10k_unigram_bigram.npz"
)

vectorizer = joblib.load(
    artifact_dir / "tfidf_10k_vectorizer.joblib"
)

metadata = pd.read_csv(
    artifact_dir / "document_metadata.csv",
    dtype={"id": str}
)

feature_names = vectorizer.get_feature_names_out()

print("Matrix:", X_tfidf.shape)
print("Format:", X_tfidf.getformat())
print("Metadata records:", len(metadata))
print("Vocabulary size:", len(feature_names))

assert X_tfidf.shape == (len(metadata), len(feature_names))

Matrix: (75110, 10000)
Format: csr
Metadata records: 75110
Vocabulary size: 10000


In [3]:
from sklearn.cluster import KMeans
from time import perf_counter

# Exploratory baseline, a starting point, not the final number of research themes
k = 10

kmeans = KMeans(
    n_clusters=k,
    init="k-means++",
    n_init=3,
    max_iter=100,
    random_state=42
)

start = perf_counter()
kmeans.fit(X_tfidf)
elapsed = perf_counter() - start

# Basic diagnostics
cluster_sizes = np.bincount(
    kmeans.labels_,
    minlength=k
)

print(f"Number of clusters: {k}")
print(f"Fitting time: {elapsed:.2f} seconds")
print(f"Iterations: {kmeans.n_iter_}")
print(f"Inertia: {kmeans.inertia_:.2f}")

print("\nCluster sizes:")
for cluster_id, size in enumerate(cluster_sizes):
    print(
        f"Cluster {cluster_id}: "
        f"{size:,} papers "
        f"({size / len(metadata) * 100:.1f}%)"
    )

Number of clusters: 10
Fitting time: 8.09 seconds
Iterations: 39
Inertia: 71273.05

Cluster sizes:
Cluster 0: 2,124 papers (2.8%)
Cluster 1: 9,965 papers (13.3%)
Cluster 2: 4,290 papers (5.7%)
Cluster 3: 23,171 papers (30.8%)
Cluster 4: 4,876 papers (6.5%)
Cluster 5: 4,053 papers (5.4%)
Cluster 6: 2,798 papers (3.7%)
Cluster 7: 3,923 papers (5.2%)
Cluster 8: 4,767 papers (6.3%)
Cluster 9: 15,143 papers (20.2%)


In [4]:
# Select the two largest clusters and the smallest cluster
largest_clusters = np.argsort(cluster_sizes)[-2:][::-1]
smallest_cluster = np.argmin(cluster_sizes)

clusters_to_inspect = [
    *largest_clusters,
    smallest_cluster
]

for c in clusters_to_inspect:
    center = kmeans.cluster_centers_[c]

    # Highest-weighted features in the centroid
    top_indices = np.argsort(center)[-12:][::-1]
    top_terms = feature_names[top_indices]

    # Documents assigned to this cluster
    members = np.flatnonzero(kmeans.labels_ == c)

    # Find documents closest to the centroid
    similarity = np.asarray(
        X_tfidf[members] @ center
    ).ravel()

    representative_indices = members[
        np.argsort(similarity)[-5:][::-1]
    ]

    print(f"\n{'=' * 65}")
    print(f"CLUSTER {c} — {len(members):,} papers")
    print(f"{'=' * 65}")

    print("\nTop centroid terms:")
    print(", ".join(top_terms))

    print("\nFive representative paper titles:")
    for i in representative_indices:
        print(f"- {metadata.iloc[i]['clean_title']}")


CLUSTER 3 — 23,171 papers

Top centroid terms:
text, data, model, models, language, based, learning, task, dataset, information, classification, using

Five representative paper titles:
- Comprehensive Implementation of TextCNN for Enhanced Collaboration between Natural Language Processing and System Recommendation
- Convolutional Neural Networks for Sentiment Analysis on Weibo Data: A Natural Language Processing Approach
- A Novel Graph-Sequence Learning Model for Inductive Text Classification
- Text classification dataset and analysis for Uzbek language
- Unsupervised Sentiment Analysis of Plastic Surgery Social Media Posts

CLUSTER 9 — 15,143 papers

Top centroid terms:
llms, models, language, llm, large language, language models, large, human, evaluation, models llms, ai, model

Five representative paper titles:
- An Empirical Study of Many-to-Many Summarization with Large Language Models
- Linguistic Intelligence in Large Language Models for Telecommunications
- Supervised Knowle